# Noncommutativity grid analysis

This notebook evaluates the empirical Sobol-profile noncommutativity metrics from
`noncommutativity_detailed.tex` across benchmark/transformation pairs that are
compatible with the current `sabench` registries.

The notebook is intentionally registry-driven. It does not maintain a manual list
of benchmark or transform keys; by default it evaluates the live benchmark and
transform registries with a small, deterministic Monte Carlo design. Environment
variables can shrink the grid for CI or fast smoke execution.

## Configuration

The default configuration evaluates all registered benchmark/transformation pairs
with a small sample size. Set `SABENCH_GRID_MAX_BENCHMARKS`,
`SABENCH_GRID_MAX_TRANSFORMS`, or `SABENCH_GRID_N_BASE` in the environment to run
a smaller smoke grid.

In [ ]:
from __future__ import annotations

import csv
import os
from collections import Counter, defaultdict
from pathlib import Path
from statistics import mean
from typing import Any

from sabench.analysis.grid import evaluate_noncommutativity_grid
from sabench.benchmarks import BENCHMARK_REGISTRY
from sabench.transforms import TRANSFORM_REGISTRY

FAST_MODE = os.environ.get("SABENCH_NOTEBOOK_FAST_MODE", "1") != "0"
N_BASE = int(os.environ.get("SABENCH_GRID_N_BASE", "128" if FAST_MODE else "4096"))
RNG_SEED = int(os.environ.get("SABENCH_GRID_SEED", "20260429"))
TAU = float(os.environ.get("SABENCH_GRID_TAU", "0.05"))
TOP_K = int(os.environ.get("SABENCH_GRID_TOP_K", "3"))
MAX_BENCHMARKS = int(os.environ.get("SABENCH_GRID_MAX_BENCHMARKS", "0"))
MAX_TRANSFORMS = int(os.environ.get("SABENCH_GRID_MAX_TRANSFORMS", "0"))
OUTPUT_DIR = Path(
    os.environ.get(
        "SABENCH_GRID_OUTPUT_DIR",
        "outputs/notebooks/02_noncommutativity_grid_analysis",
    )
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

benchmark_keys = tuple(BENCHMARK_REGISTRY)
transform_keys = tuple(TRANSFORM_REGISTRY)
if MAX_BENCHMARKS > 0:
    benchmark_keys = benchmark_keys[:MAX_BENCHMARKS]
if MAX_TRANSFORMS > 0:
    transform_keys = transform_keys[:MAX_TRANSFORMS]

print(
    f"Configured grid with {len(benchmark_keys)} benchmarks, "
    f"{len(transform_keys)} transforms, N_BASE={N_BASE}, TAU={TAU}."
)

## Execute the registry-driven grid

Each result row represents one benchmark/transformation pair. Pair-level failures
are retained as status rows rather than stopping the full grid.

In [ ]:
results = evaluate_noncommutativity_grid(
    benchmark_keys=benchmark_keys,
    transform_keys=transform_keys,
    n_base=N_BASE,
    seed=RNG_SEED,
    tau=TAU,
    top_k=TOP_K,
)
rows = [result.as_dict() for result in results]

for row in rows:
    if row.get("decision_score_s1") is not None:
        row["D_s1"] = row["decision_score_s1"]
        row["delta_s1"] = row["sensitivity_shift_s1"]
        row["D_st"] = row["decision_score_st"]
        row["delta_st"] = row["sensitivity_shift_st"]
        row["threshold_flip_s1"] = row["threshold_flip_count_s1"]
        row["threshold_flip_st"] = row["threshold_flip_count_st"]
        row["top_driver_y_s1"] = row["top_source_index_s1"]
        row["top_driver_z_s1"] = row["top_transformed_index_s1"]
        row["top_driver_y_st"] = row["top_source_index_st"]
        row["top_driver_z_st"] = row["top_transformed_index_st"]

status_counts = Counter(row["metrics_status"] for row in rows)
print(f"Evaluated {len(rows)} candidate pairs")
print(dict(sorted(status_counts.items())))

## Export pair-level tables

The exported tables are the durable artifacts for downstream review and plotting.
`pair_status.csv` contains every candidate pair. `noncommutativity_metrics.csv`
contains only rows with computed Sobol-profile metrics.

In [ ]:
PAIR_STATUS_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "pair_status",
    "metrics_status",
    "reason",
    "benchmark_output_kind",
    "transform_mechanism",
    "transform_tags",
    "n_base",
    "seed",
    "n_inputs",
    "n_evaluations",
    "raw_output_shape",
    "transformed_output_shape",
    "raw_output_finite",
    "transformed_output_finite",
    "raw_variance",
    "transformed_variance",
]
METRIC_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "D_s1",
    "delta_s1",
    "D_st",
    "delta_st",
    "threshold_flip_s1",
    "threshold_flip_st",
    "topk_changed_s1",
    "topk_changed_st",
    "max_abs_shift_s1",
    "max_abs_shift_st",
    "mean_abs_shift_s1",
    "mean_abs_shift_st",
    "spearman_s1",
    "spearman_st",
    "top_driver_y_s1",
    "top_driver_z_s1",
    "top_driver_y_st",
    "top_driver_z_st",
]

computed_rows = [row for row in rows if row["metrics_status"] == "computed"]
metrics_rows = [
    {column: row.get(column) for column in METRIC_COLUMNS}
    for row in computed_rows
]

def write_csv(path: Path, table_rows: list[dict[str, Any]], columns: list[str]) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(table_rows)

write_csv(OUTPUT_DIR / "pair_status.csv", rows, PAIR_STATUS_COLUMNS)
write_csv(OUTPUT_DIR / "noncommutativity_metrics.csv", metrics_rows, METRIC_COLUMNS)

print(f"Wrote {OUTPUT_DIR / 'pair_status.csv'}")
print(f"Wrote {OUTPUT_DIR / 'noncommutativity_metrics.csv'}")

## Summaries by transform and benchmark

These summaries rank transformations and benchmarks by the empirical size of the
Sobol-profile perturbation.

In [ ]:
SUMMARY_METRICS = ["D_s1", "delta_s1", "D_st", "delta_st"]


def _float_values(table_rows: list[dict[str, Any]], key: str) -> list[float]:
    values: list[float] = []
    for row in table_rows:
        value = row.get(key)
        if value is None or value == "":
            continue
        values.append(float(value))
    return values


def summarize_by(group_key: str) -> list[dict[str, Any]]:
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in computed_rows:
        grouped[str(row[group_key])].append(row)

    summary_rows: list[dict[str, Any]] = []
    for key, group_rows in sorted(grouped.items()):
        summary: dict[str, Any] = {group_key: key, "n_pairs_computed": len(group_rows)}
        for metric in SUMMARY_METRICS:
            values = _float_values(group_rows, metric)
            summary[f"mean_{metric}"] = mean(values) if values else None
            summary[f"max_{metric}"] = max(values) if values else None
        summary_rows.append(summary)
    return summary_rows

summary_columns = [
    "n_pairs_computed",
    "mean_D_s1",
    "max_D_s1",
    "mean_delta_s1",
    "max_delta_s1",
    "mean_D_st",
    "max_D_st",
    "mean_delta_st",
    "max_delta_st",
]
summary_by_transform = summarize_by("transform_key")
summary_by_benchmark = summarize_by("benchmark_key")
write_csv(
    OUTPUT_DIR / "summary_by_transform.csv",
    summary_by_transform,
    ["transform_key", *summary_columns],
)
write_csv(
    OUTPUT_DIR / "summary_by_benchmark.csv",
    summary_by_benchmark,
    ["benchmark_key", *summary_columns],
)

print(f"Wrote {OUTPUT_DIR / 'summary_by_transform.csv'}")
print(f"Wrote {OUTPUT_DIR / 'summary_by_benchmark.csv'}")

## Quick interpretation checks

The output tables should be interpreted as empirical diagnostics. The Decision
Score `D` captures soft threshold-crossing behavior near `tau`, while Sensitivity
Shift `delta` captures Bray-Curtis profile movement. A large value in either
metric indicates that the transformation materially changed the benchmark's
estimated Sobol sensitivity profile.

In [ ]:
if computed_rows:
    largest_delta_s1 = max(computed_rows, key=lambda row: float(row.get("delta_s1") or 0.0))
    largest_delta_st = max(computed_rows, key=lambda row: float(row.get("delta_st") or 0.0))
    print("Largest first-order sensitivity shift:")
    print(
        largest_delta_s1["benchmark_key"],
        largest_delta_s1["transform_key"],
        largest_delta_s1["delta_s1"],
    )
    print("Largest total-effect sensitivity shift:")
    print(
        largest_delta_st["benchmark_key"],
        largest_delta_st["transform_key"],
        largest_delta_st["delta_st"],
    )
else:
    print("No compatible pairs produced computed metrics in this run.")